# Spotify Hit Predictor — SQL Exploration

**Objective:** query the cleaned data set with SQL to find patterns in popularity, before modeling. There is where H1 ("a dectectable pattern exists") and H2("genre is secondary") 
get their first test.

**Input:** `tracks_clean.csv` — 112,782 tracks, 114 genres (output of notebook
01, where cleaning and validation are documented).

**This notebook produces:** genre popularity rankings, a hit-rate analysis by
genre (popularity ≥ 70 threshold), and a cross-genre duplicate count flagged
as a data-leakage risk for the modeling phase.

In [1]:
import pandas as pd 
import sqlite3

df = pd.read_csv('../data/tracks_clean.csv')
print("Numbers of songs loaded:", df.shape[0])


Numbers of songs loaded: 112782


## Loading the data into SQLite

The cleaned dataset (`tracks_clean.csv`) is loaded into a SQLite database so
the analysis can be done in SQL

In [2]:
# Connect to the database 
conn = sqlite3.connect('spotify.db')

# Cleanned DataFrame put into a 'tracks' table
df.to_sql('tracks', conn, if_exists='replace', index='False')
print('tracks table created in spotify.db' )

tracks table created in spotify.db


In [3]:
query = 'SELECT COUNT(*) AS total_songs FROM tracks'
pd.read_sql(query, conn)

,total_songs
0,112782


## Q8 — Which genres have the highest average popularity?

Grouping by genre and averaging popularity. `MAX(popularity)` is included to
catch the average-vs-peak paradox: a genre can rank low on average yet contain
the single most popular track.

In [4]:
query = """ 
SELECT track_genre, 
     COUNT(*) AS songs,
     ROUND(AVG(popularity), 1) AS popularity_avg,
     MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre
ORDER BY popularity_avg DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,popularity_avg,popularity_max
0,pop-film,998,59.3,80
1,k-pop,993,57.0,88
2,chill,999,53.7,93
3,sad,1000,52.4,83
4,grunge,999,49.6,85
5,indian,993,49.5,88
6,anime,999,48.8,83
7,emo,999,48.1,87
8,sertanejo,1000,47.9,63
9,pop,993,47.9,100


**Result:** pop-film (59.3), k-pop (57.0), chill (53.7) lead. The spread across
the top genres is narrow — about 15 points.

**Finding (H2):** if genre were a strong predictor of popularity, we'd expect a
wide gap between "popular" and "unpopular" genres. Instead the averages cluster
tightly. Early evidence that the genre label carries limited predictive signal.

### Inspecting the 'piano' genre

Checking what actually sits inside a non-standard genre label. `piano` is an
instrument, not a genre, this confirms theinconsistency documented
in notebook 01 (Q7).

In [5]:
query = """
SELECT track_name, artists, popularity, duration_ms/60000.0 AS min
FROM tracks
WHERE track_genre = 'piano'
ORDER BY popularity DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,track_name,artists,popularity,min
0,I Ain't Worried,OneRepublic,96,2.474750
1,Running Up That Hill (A Deal With God),Kate Bush,90,4.982217
2,Hold Me Closer,Elton John;Britney Spears,89,3.370750
3,Somewhere Only We Know,Keane,85,3.952433
4,Running Up That Hill (A Deal With God) - 2018 ...,Kate Bush,85,5.014000
5,I'm Still Standing,Elton John,84,3.057333
6,Counting Stars,OneRepublic,83,4.287767
7,Sunshine,OneRepublic,83,2.730900
8,"Rocket Man (I Think It's Going To Be A Long, L...",Elton John,81,4.693550
9,How to Save a Life,The Fray,80,4.375550


**Result:** the top 'piano' tracks are pop and rock hits.
OneRepublic's "I Ain't Worried", Kate Bush's "Running Up That Hill", Billy Joel's "Uptown Girl", multiple Elton John songs. Not one is an instrumental piano piece.
These are vocal mainstream songs that happen to feature piano in the arrangement.

**Finding (H2):** 'piano' is an instrument label, not a genre. A pop song and a rock song sit side by side under it because they share an instrument, not a
sound category. This is direct evidence that track_genre mixes taxonomies — the same inconsistency documented in notebook 01 (Q7), now shown concretely.
If the label can't even separate pop from rock, it's a weak predictive variable by construction.

## Q10 — Unique songs vs total rows (cross-genre duplicates)

Notebook 01 flagged that duplicates were likely the same track appearing under multiple genres. This query confirms it directly: GROUP_CONCAT lists every
genre each track_id is tagged with.

In [6]:
query = """
SELECT track_name, artists, COUNT(*) AS count, 
       GROUP_CONCAT(track_genre, ', ') AS genres
FROM tracks
GROUP BY track_id
HAVING count > 1
ORDER BY count DESC
LIMIT 15
"""
pd.read_sql(query, conn)

,track_name,artists,count,genres
0,Baby Blue - Remastered 2010,Badfinger,9,"blues, country, folk, j-pop, j-rock, power-pop..."
1,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc..."
2,Layla,Derek & The Dominos,8,"blues, british, country, folk, hard-rock, psyc..."
3,Liggi,Ritviz,7,"edm, hip-hop, indian, indie-pop, indie, pop-fi..."
4,Mountain Song,Jane's Addiction,7,"alt-rock, blues, funk, grunge, hard-rock, meta..."
5,Trouble No More,Allman Brothers Band,7,"blues, country, folk, hard-rock, j-rock, singe..."
6,Let Me Hear,"Fear, and Loathing in Las Vegas",7,"hard-rock, hardcore, j-pop, j-rock, metal, met..."
7,Never Gonna Give You Up,The Black Keys,7,"alt-rock, alternative, blues, garage, punk-roc..."
8,Udd Gaye,Ritviz,7,"edm, hip-hop, indian, indie-pop, indie, pop-fi..."
9,Show Me The Way,Peter Frampton,7,"blues, british, country, folk, hard-rock, sing..."


**Result:** the most-tagged tracks appear under 7–9 genres each. A single recording is labeled across genres that share almost nothing sonically.

This is the mechanism behind the 21% duplicate rate (89,023 unique tracks in112,782 rows): one recording, many genre tags.

**Dual reading of this artifact:**
- **As business signal (Q24):** a track tagged across more genres has more
  playlist surface — more chances to be surfaced by Spotify's collaborative
  filtering. The versatility analysis will use these duplicate rows on purpose.
- **As technical risk (Week 4):** the same recording landing in both the train
  and test split would inflate the model's apparent accuracy. These duplicates
  will be removed before training (data leakage prevention).

Same artifact, opposite treatment — signal for the versatility question, noise for the model.

**Finding (H2):** if one recording carries 9 contradictory genre labels, the label cannot be describing the sound. The measurable audio is the more reliable
variable — which is exactly what H2 predicts.

Now the scale. The previous query showed *which* tracks repeat; this one counts *how many* across the whole table.

In [7]:
query = """ 
SELECT COUNT(*) AS total_rows ,
       COUNT(DISTINCT track_id) AS unique_songs,
       COUNT(*) - COUNT(DISTINCT track_id) AS duplicates 
FROM tracks 
"""

pd.read_sql(query, conn)

,total_rows,unique_songs,duplicates
0,112782,89023,23759


**Result:** 112,782 total rows, 89,023 unique songs → 23,759 duplicate rows (21% of the dataset).

## Q11, Q14 — Hit rate by genre, and defining the "hit" threshold

A "hit" needs a popularity cutoff, and that choice shouldn't be arbitrary. This counts hits per genre at threshold 70. The same query is then re-run at
60 (below) to test whether the ranking is stable. 

In [8]:
query = """
SELECT track_genre, 
      COUNT(*) AS songs, 
      SUM(CASE WHEN popularity >= 70 THEN 1 ELSE 0 END) AS hits,
      MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre 
ORDER BY hits DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,hits,popularity_max
0,pop,993,317,100
1,dance,965,244,100
2,electro,998,241,89
3,k-pop,993,225,88
4,house,999,221,90
5,metal,995,217,88
6,rock,1000,198,96
7,indie,997,185,92
8,edm,993,181,98
9,indie-pop,1000,177,88


In [9]:
query = """
SELECT track_genre, 
      COUNT(*) AS songs, 
      SUM(CASE WHEN popularity >= 60 THEN 1 ELSE 0 END) AS hits,
      MAX(popularity) AS popularity_max
FROM tracks 
GROUP BY track_genre 
ORDER BY hits DESC
LIMIT 15
"""

pd.read_sql(query, conn)


,track_genre,songs,hits,popularity_max
0,pop,993,644,100
1,pop-film,998,530,80
2,k-pop,993,502,88
3,metal,995,471,88
4,electro,998,450,89
5,house,999,411,90
6,hip-hop,990,408,99
7,edm,993,376,98
8,hard-rock,998,360,88
9,indie-pop,1000,352,88


**Result (threshold 70):** pop leads with 317 hits, followed by dance (244) and electro (241). The dataset is balanced by genre and with ~1,000 songs per genre, so hit counts are directly comparable
without normalizing.

## Q9 — Does 'pop' dominate?
It depends entirely on how we measure.

- **By average popularity (Q8):** pop ranks #9. Mediocre.
- **By number of hits (Q11):** pop ranks #1 — 317 hits, and the only track scoring 100.

Both are true. Pop is a volume factory: it releases so many songs that most are forgettable (dragging the average down to #9) yet it still produces more
hits in absolute terms than any other genre. A niche genre like k-pop is more consistent (higher average) but generates fewer absolute hits.

**Finding (H2):** the genre label "pop" simultaneously means the single best track in the dataset *and* a mass of mediocre ones. It cannot predict whether
*a given song* will be a hit — it's too broad and internally contradictory.
This is precisely why the model should lean on measurable audio features
rather than the genre tag.

In [10]:
query = """
SELECT track_name, artists, track_genre, popularity
FROM tracks
WHERE popularity = 100
"""

pd.read_sql(query, conn)

,track_name,artists,track_genre,popularity
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,dance,100
1,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,pop,100


**Note:** "pop has 317 hits" means 317 hit-rows under the pop label not 317 unique songs. Given the 21% cross-genre duplication (Q10),
some of these hits are the same recording counted under multiple genres, as "Unholy" is. The ranking holds, but the honest phrasing is "hit-rows per
label," not "unique hits per genre."